# Customer Churn Prediction — Step 6: Multiple Model Training and Validation Comparison

This notebook implements **Step 6: Multiple Model Training and Validation Comparison** for the Customer Churn Prediction project.

### Pipeline Stage & Scope:
```text
Preprocessed Training Data (data/processed/train.csv)
        ↓
Candidate Classifiers:
    ├── 1. Logistic Regression (Baseline: max_iter=1000, random_state=42)
    ├── 2. Decision Tree (DecisionTreeClassifier: random_state=42)
    └── 3. Random Forest (RandomForestClassifier: n_estimators=100, random_state=42, n_jobs=-1)
        ↓
Model Training (.fit() on X_train only)
        ↓
Validation Evaluation (.predict(), .predict_proba() on X_val only)
        ↓
Metrics Comparison (Accuracy, Precision, Recall, F1 Score, ROC-AUC)
        ↓
Confusion Matrix & Error Trade-off Analysis
        ↓
Overfitting Diagnostics (Train vs. Validation Performance Gap)
        ↓
Random Forest Feature Importance & Non-Causal Interpretation
        ↓
Validation Champion Selection & Candidate Artifact Serialization
```

> [!IMPORTANT]
> **STRICT TEST SET PROTECTION**: The test set (`data/processed/test.csv`) remains completely untouched, unread, and locked. All model comparisons and architectural selection decisions are performed exclusively on the **validation set** (`data/processed/val.csv`).

## 1. Candidate Models & Inductive Biases

In this stage, we compare three distinct model families to understand how linear assumptions compare against non-linear hierarchical partitioners and bagging ensembles:

### 1. Logistic Regression (Baseline)
- **Inductive Bias**: Assumes a linear relationship between input features and the log-odds of churn: $\log\left(\frac{p}{1-p}\right) = \mathbf{w}^T \mathbf{x} + b$.
- **Strengths**: Convex optimization guarantee (global optimum), highly calibrated probability estimates, low computational complexity, robust against variance on moderately sized tabular data.
- **Limitations**: Cannot model feature interactions or non-linear thresholds unless explicitly engineered into the feature space.

### 2. Decision Tree (`DecisionTreeClassifier`)
- **Inductive Bias**: Recursive binary partitioning of feature space using axis-aligned orthogonal hyperplanes that maximize Gini impurity reduction.
- **Strengths**: Captures complex non-linear relationships and high-order feature interactions without requiring manual transformations.
- **Limitations**: Highly sensitive to small variations in training data (high variance). Without tree depth or leaf sample constraints, default decision trees rapidly overfit and memorize training samples.

### 3. Random Forest (`RandomForestClassifier`)
- **Inductive Bias**: Bagging (Bootstrap Aggregating) ensemble of decorrelated deep decision trees combined with random subspace feature sampling at each split.
- **Strengths**: Substantially reduces variance compared to individual decision trees through ensemble averaging: $\text{Var}(\bar{X}) = \rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$. Resilient to outliers and provides built-in feature importance estimates.
- **Limitations**: Ensembles unpruned trees by default which can still exhibit noticeable generalization gaps on noisy tabular data before proper regularization and depth tuning.

## 2. Environment Setup & Data Loading

We load project configurations and helper modules from `src`.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DECISION_TREE_CM_PATH,
    DECISION_TREE_MODEL_PATH,
    FEATURE_IMPORTANCE_FIGURE_PATH,
    LOGISTIC_REGRESSION_CM_PATH,
    MODEL_COMPARISON_FIGURE_PATH,
    PROCESSED_DATA_DIR,
    RANDOM_FOREST_CM_PATH,
    RANDOM_FOREST_MODEL_PATH,
    RANDOM_STATE,
    TARGET_COL,
)
from src.evaluation.metrics import (
    calculate_classification_metrics,
    compare_models,
    compute_confusion_matrix,
    get_confusion_matrix_breakdown,
)
from src.evaluation.plots import (
    plot_confusion_matrix,
    plot_feature_importance,
    plot_model_comparison,
)
from src.models.model import (
    create_decision_tree_model,
    create_logistic_regression_model,
    create_random_forest_model,
    get_model_feature_importances,
    load_model,
    predict_churn,
    predict_churn_probabilities,
    save_model,
    train_model,
)

train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DATA_DIR / "val.csv")

print(
    f"Training dataset shape:   {train_df.shape} (rows: {len(train_df):,}, cols: {train_df.shape[1]})"
)
print(
    f"Validation dataset shape: {val_df.shape} (rows: {len(val_df):,}, cols: {val_df.shape[1]})"
)

Training dataset shape:   (4930, 46) (rows: 4,930, cols: 46)
Validation dataset shape: (1056, 46) (rows: 1,056, cols: 46)


## 3. Feature Matrix & Target Label Separation

We separate feature inputs ($X$) and churn labels ($y$) for the training and validation partitions. We confirm that the test set remains locked.

In [2]:
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df.drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]
feature_names = list(X_train.columns)

print(f"X_train feature matrix: {X_train.shape}, y_train labels: {y_train.shape}")
print(f"X_val feature matrix:   {X_val.shape}, y_val labels:   {y_val.shape}")

print(
    f"\nTraining churn rate:   {y_train.mean() * 100:.2f}% (Churn=1: {(y_train == 1).sum():,}, Churn=0: {(y_train == 0).sum():,})"
)
print(
    f"Validation churn rate: {y_val.mean() * 100:.2f}% (Churn=1: {(y_val == 1).sum():,}, Churn=0: {(y_val == 0).sum():,})"
)
print("\nTest set status: UNTOUCHED (Locked until Step 7 final evaluation).")

X_train feature matrix: (4930, 45), y_train labels: (4930,)
X_val feature matrix:   (1056, 45), y_val labels:   (1056,)

Training churn rate:   26.53% (Churn=1: 1,308, Churn=0: 3,622)
Validation churn rate: 26.52% (Churn=1: 280, Churn=0: 776)

Test set status: UNTOUCHED (Locked until Step 7 final evaluation).


## 4. Train Model 1: Logistic Regression (Baseline)

We instantiate and train the baseline Logistic Regression model using `create_logistic_regression_model()`.

In [3]:
lr_model = create_logistic_regression_model(random_state=RANDOM_STATE)
train_model(lr_model, X_train, y_train)

y_val_pred_lr = predict_churn(lr_model, X_val)
y_val_proba_lr = predict_churn_probabilities(lr_model, X_val)
metrics_lr = calculate_classification_metrics(y_val, y_val_pred_lr, y_val_proba_lr)

print("Logistic Regression validation metrics:")
print(metrics_lr)

Logistic Regression validation metrics:
{'accuracy': 0.8059, 'precision': 0.6459, 'recall': 0.5929, 'f1': 0.6182, 'roc_auc': 0.8454}


## 5. Train Model 2: Decision Tree

We instantiate and train an unconstrained Decision Tree classifier using `create_decision_tree_model(random_state=42)`.

In [4]:
dt_model = create_decision_tree_model(random_state=RANDOM_STATE)
train_model(dt_model, X_train, y_train)

y_val_pred_dt = predict_churn(dt_model, X_val)
y_val_proba_dt = predict_churn_probabilities(dt_model, X_val)
metrics_dt = calculate_classification_metrics(y_val, y_val_pred_dt, y_val_proba_dt)

print("Decision Tree validation metrics:")
print(metrics_dt)

Decision Tree validation metrics:
{'accuracy': 0.7178, 'precision': 0.469, 'recall': 0.4857, 'f1': 0.4772, 'roc_auc': 0.643}


## 6. Train Model 3: Random Forest

We instantiate and train the Random Forest ensemble using `create_random_forest_model(n_estimators=100, random_state=42, n_jobs=-1)`.

In [5]:
rf_model = create_random_forest_model(
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
)
train_model(rf_model, X_train, y_train)

y_val_pred_rf = predict_churn(rf_model, X_val)
y_val_proba_rf = predict_churn_probabilities(rf_model, X_val)
metrics_rf = calculate_classification_metrics(y_val, y_val_pred_rf, y_val_proba_rf)

print("Random Forest validation metrics:")
print(metrics_rf)

Random Forest validation metrics:
{'accuracy': 0.7756, 'precision': 0.5939, 'recall': 0.4857, 'f1': 0.5344, 'roc_auc': 0.8131}


## 7. Model Performance Comparison Table

We compile the validation metrics across all three candidate models into a consolidated comparison DataFrame using `compare_models()`.

In [6]:
models_dict = {
    "Logistic Regression": metrics_lr,
    "Decision Tree": metrics_dt,
    "Random Forest": metrics_rf,
}

comparison_df = compare_models(models_dict)
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.8059,0.6459,0.5929,0.6182,0.8454
1,Decision Tree,0.7178,0.4690,0.4857,0.4772,0.6430
2,Random Forest,0.7756,0.5939,0.4857,0.5344,0.8131


## 8. Grouped Bar Chart Visualization

We visualize the performance across Accuracy, Precision, Recall, F1 Score, and ROC-AUC using `plot_model_comparison()`.

In [7]:
fig_comp = plot_model_comparison(
    comparison_df,
    title="Validation Performance Comparison Across Candidate Models",
    save_path=MODEL_COMPARISON_FIGURE_PATH,
)
plt.show()

## 9. Validation Confusion Matrices & Error Trade-off Analysis

We examine the breakdown of True Negatives, False Positives, False Negatives, and True Positives across all three candidate models on the 1,056 validation samples.

In [8]:
cm_lr = compute_confusion_matrix(y_val, y_val_pred_lr)
cm_dt = compute_confusion_matrix(y_val, y_val_pred_dt)
cm_rf = compute_confusion_matrix(y_val, y_val_pred_rf)

breakdown_lr = get_confusion_matrix_breakdown(y_val, y_val_pred_lr)
breakdown_dt = get_confusion_matrix_breakdown(y_val, y_val_pred_dt)
breakdown_rf = get_confusion_matrix_breakdown(y_val, y_val_pred_rf)

print("Confusion Matrix Breakdown (1,056 validation samples):\n")
print(
    f"Logistic Regression: TN={breakdown_lr['tn']} (64.87%), FP={breakdown_lr['fp']} (8.62%), FN={breakdown_lr['fn']} (10.80%), TP={breakdown_lr['tp']} (15.72%)"
)
print(
    f"Decision Tree:       TN={breakdown_dt['tn']} (58.90%), FP={breakdown_dt['fp']} (14.58%), FN={breakdown_dt['fn']} (13.64%), TP={breakdown_dt['tp']} (12.88%)"
)
print(
    f"Random Forest:       TN={breakdown_rf['tn']} (64.68%), FP={breakdown_rf['fp']} (8.81%), FN={breakdown_rf['fn']} (13.64%), TP={breakdown_rf['tp']} (12.88%)"
)

Confusion Matrix Breakdown (1,056 validation samples):

Logistic Regression: TN=685 (64.87%), FP=91 (8.62%), FN=114 (10.80%), TP=166 (15.72%)
Decision Tree:       TN=622 (58.90%), FP=154 (14.58%), FN=144 (13.64%), TP=136 (12.88%)
Random Forest:       TN=683 (64.68%), FP=93 (8.81%), FN=144 (13.64%), TP=136 (12.88%)


In [9]:
# Plot and save confusion matrices for all 3 models
fig_cm_lr = plot_confusion_matrix(
    cm_lr,
    title="Validation Confusion Matrix — Logistic Regression Baseline",
    save_path=LOGISTIC_REGRESSION_CM_PATH,
)
plt.show()

fig_cm_dt = plot_confusion_matrix(
    cm_dt,
    title="Validation Confusion Matrix — Decision Tree Baseline",
    save_path=DECISION_TREE_CM_PATH,
)
plt.show()

fig_cm_rf = plot_confusion_matrix(
    cm_rf,
    title="Validation Confusion Matrix — Random Forest Baseline",
    save_path=RANDOM_FOREST_CM_PATH,
)
plt.show()

## 10. Train vs. Validation Overfitting Diagnostics

A fundamental danger in machine learning is **overfitting** (high variance), where a complex model memorizes idiosyncratic training noise rather than learning generalizable decision boundaries.

We evaluate each model on both `X_train` and `X_val` to diagnose the generalization gap.

In [10]:
# Compute training predictions and metrics
y_train_pred_lr = predict_churn(lr_model, X_train)
y_train_pred_dt = predict_churn(dt_model, X_train)
y_train_pred_rf = predict_churn(rf_model, X_train)

train_m_lr = calculate_classification_metrics(y_train, y_train_pred_lr)
train_m_dt = calculate_classification_metrics(y_train, y_train_pred_dt)
train_m_rf = calculate_classification_metrics(y_train, y_train_pred_rf)

overfit_df = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression",
            "Train Accuracy": train_m_lr["accuracy"],
            "Val Accuracy": metrics_lr["accuracy"],
            "Train F1": train_m_lr["f1"],
            "Val F1": metrics_lr["f1"],
            "F1 Gap": f"{metrics_lr['f1'] - train_m_lr['f1']:+.4f}",
        },
        {
            "Model": "Decision Tree",
            "Train Accuracy": train_m_dt["accuracy"],
            "Val Accuracy": metrics_dt["accuracy"],
            "Train F1": train_m_dt["f1"],
            "Val F1": metrics_dt["f1"],
            "F1 Gap": f"{metrics_dt['f1'] - train_m_dt['f1']:+.4f}",
        },
        {
            "Model": "Random Forest",
            "Train Accuracy": train_m_rf["accuracy"],
            "Val Accuracy": metrics_rf["accuracy"],
            "Train F1": train_m_rf["f1"],
            "Val F1": metrics_rf["f1"],
            "F1 Gap": f"{metrics_rf['f1'] - train_m_rf['f1']:+.4f}",
        },
    ]
)
overfit_df

,Model,Train Accuracy,Val Accuracy,Train F1,Val F1,F1 Gap
0,Logistic Regression,0.8037,0.8059,0.5980,0.6182,+0.0202
1,Decision Tree,0.9980,0.7178,0.9962,0.4772,-0.5190
2,Random Forest,0.9980,0.7756,0.9962,0.5344,-0.4618


### Diagnostic Findings on Overfitting:
1. **Decision Tree exhibits massive overfitting**: Train F1 is 0.9962 (99.80% accuracy) while Validation F1 plummets to 0.4772 (71.78% accuracy). The $-0.5190$ generalization gap occurs because an unconstrained tree expands until leaves are pure, essentially storing training examples as lookup rules.
2. **Random Forest moderates variance but remains unregularized**: Train F1 is 0.9962 vs. Validation F1 of 0.5344. By averaging 100 decorrelated trees, Random Forest significantly improves validation ROC-AUC (0.8131 vs 0.6430 for DT). However, without `max_depth` limits or `min_samples_leaf` constraints, individual trees overfit their bootstrap bags.
3. **Logistic Regression shows exceptional generalization stability**: Train F1 (0.5980) closely tracks Validation F1 (0.6182), demonstrating virtually zero variance error. The linear parametric inductive bias prevents memorization of noise.

## 11. Feature Importance Analysis (Random Forest)

We extract Gini feature importances from the Random Forest model using `get_model_feature_importances()` and plot the top 15 predictors using `plot_feature_importance()`.

In [11]:
rf_importances = get_model_feature_importances(rf_model, feature_names)

print("Top 10 Feature Importances (Random Forest):")
for feat, imp in rf_importances.head(10).items():
    print(f"{feat:<36} {imp:.4f} ({imp * 100:.1f}%)")

Top 10 Feature Importances (Random Forest):
TotalCharges                         0.1600 (16.0%)
tenure                               0.1362 (13.6%)
MonthlyCharges                       0.1353 (13.5%)
Contract_Month-to-month              0.0503 (5.0%)
PaymentMethod_Electronic check       0.0337 (3.4%)
InternetService_Fiber optic          0.0306 (3.1%)
gender_Male                          0.0249 (2.5%)
gender_Female                        0.0247 (2.5%)
OnlineSecurity_No                    0.0240 (2.4%)
TechSupport_No                       0.0226 (2.3%)
Name: importance, dtype: float64


In [12]:
fig_fi = plot_feature_importance(
    rf_importances,
    top_n=15,
    title="Top 15 Feature Importances — Random Forest (Validation Comparison)",
    save_path=FEATURE_IMPORTANCE_FIGURE_PATH,
)
plt.show()

## 12. Critical Nuance: Feature Importance vs. Causality

> [!CAUTION]
> **FEATURE IMPORTANCE IS PREDICTIVE, NOT CAUSAL**:
> Random Forest feature importances reflect **Mean Decrease in Impurity (Gini importance)** across tree splits. A high importance indicates that the feature is frequently selected and effectively partitions churn vs. non-churn customer subgroups. 
>
> It does **NOT** imply that manipulating the feature will causally prevent churn:
> 1. **High TotalCharges and tenure** (together accounting for ~29.6% of importance) are strong indicators of account maturity; customers who haven't churned naturally accumulate high tenure and charges.
> 2. **Continuous features bias Gini splits**: Tree algorithms inherently favor features with high cardinality (many distinct cutpoints) over binary categorical indicators.
> 3. **Correlation without Intervention**: Forcing a customer onto an electronic check payment method or raising their monthly charges will not causally change their loyalty without addressing underlying product satisfaction.

## 13. Model Selection Analysis & Engineering Decision

### Validation Performance Synthesis:
| Criterion | Logistic Regression (Baseline) | Decision Tree | Random Forest | Winner |
|:---|:---:|:---:|:---:|:---:|
| **Accuracy** | **0.8059** | 0.7178 | 0.7756 | **Logistic Regression** |
| **Precision** | **0.6459** | 0.4690 | 0.5939 | **Logistic Regression** |
| **Recall** | **0.5929** | 0.4857 | 0.4857 | **Logistic Regression** |
| **F1 Score** | **0.6182** | 0.4772 | 0.5344 | **Logistic Regression** |
| **ROC-AUC** | **0.8454** | 0.6430 | 0.8131 | **Logistic Regression** |
| **Missed Churners (FN)** | **114 (10.8%)** | 144 (13.6%) | 144 (13.6%) | **Logistic Regression** |
| **Overfitting (F1 Gap)** | **+0.0202 (Stable)** | -0.5190 (Severe) | -0.4618 (High) | **Logistic Regression** |

### Why Does Logistic Regression Win at the Baseline Stage?
1. **Absence of Hyperparameter Regularization**: Tree-based models require structural constraints (`max_depth`, `min_samples_split`, `min_samples_leaf`) to restrict growth and prevent memorization. Out-of-the-box unpruned trees suffer from high variance on noisy tabular churn data.
2. **Effective Standardization**: Step 4 scaled continuous features and one-hot encoded categories with unknown handling, providing optimal geometric conditioning for Logistic Regression's convex log-loss solver.
3. **Threshold Calibration**: At the default 0.5 threshold, Logistic Regression captures 166 true churners (Recall 59.29%) compared to only 136 for Decision Tree and Random Forest (Recall 48.57%).

## 14. Save Candidate Model Artifacts

We serialize the fitted Decision Tree and Random Forest candidate models to `models/` using `save_model()` and verify reproducibility via `load_model()`.

In [13]:
# Serialize fitted candidate models
saved_dt_path = save_model(dt_model, DECISION_TREE_MODEL_PATH)
saved_rf_path = save_model(rf_model, RANDOM_FOREST_MODEL_PATH)

print(
    f"Saved Decision Tree model artifact to: {saved_dt_path.relative_to(PROJECT_ROOT)}"
)
print(
    f"Saved Random Forest model artifact to: {saved_rf_path.relative_to(PROJECT_ROOT)}"
)

# Verification through deserialization
loaded_dt = load_model(saved_dt_path)
loaded_rf = load_model(saved_rf_path)

assert np.array_equal(predict_churn(dt_model, X_val), predict_churn(loaded_dt, X_val))
assert np.array_equal(predict_churn(rf_model, X_val), predict_churn(loaded_rf, X_val))

print("\nVerification: Loaded DT predictions match original: True")
print("Verification: Loaded RF predictions match original: True")

Saved Decision Tree model artifact to: models/decision_tree_baseline.joblib
Saved Random Forest model artifact to: models/random_forest_baseline.joblib

Verification: Loaded DT predictions match original: True
Verification: Loaded RF predictions match original: True


## 15. Summary & Transition to Step 7

### Key Takeaways from Step 6:
1. **Validation Champion**: **Logistic Regression** remains the top-performing model on the validation dataset across all 5 evaluation metrics (Accuracy: 0.8059, Precision: 0.6459, Recall: 0.5929, F1: 0.6182, ROC-AUC: 0.8454).
2. **Ensemble Variance Reduction**: While Decision Tree performed poorly (F1: 0.4772, ROC-AUC: 0.6430), Random Forest recovered substantial predictive capacity (F1: 0.5344, ROC-AUC: 0.8131) confirming bagging's ability to reduce variance.
3. **Pruning Necessity**: Both tree architectures severely overfitted the training data (Train F1 ~0.9962). Regularization (tuning tree depth, minimum leaf samples) is necessary for tree models to compete with or surpass linear baselines.
4. **Feature Importance Insights**: Account duration (`tenure`), financial commitments (`TotalCharges`, `MonthlyCharges`), and contractual commitment (`Contract_Month-to-month`) are the most informative signals for tree partitioning.
5. **Test Set Integrity Preserved**: The test set was untouched throughout model training, comparison, and diagnostics.

### Next Steps (Step 7):
- Hyperparameter tuning (GridSearchCV / RandomizedSearchCV) for tree depth, regularization, and estimators.
- Class imbalance handling experiments (class weighting, SMOTE, threshold tuning).
- Final model selection and unbiased test set evaluation on `data/processed/test.csv`.